# TP_S1 — IA Symbolique : détecter des intrusions avec des règles
**Cours IA & Cybersécurité — Master 1**

---

## Ce que vous allez faire

Avant le Machine Learning, l'IA raisonnait avec des **règles écrites par des experts humains**.  
C'est encore très utilisé aujourd'hui dans les SIEM, les firewalls et les systèmes de détection d'intrusion (IDS).

Dans ce TP, vous allez construire **trois systèmes de détection d'intrusion** de complexité croissante :

| Partie | Concept | Ce que vous construisez |
|--------|---------|-------------------------|
| 1 | Logique booléenne | Conditions simples sur du trafic réseau |
| 2 | Règles SI...ALORS | Système expert à base de règles |
| 3 | Arbre de décision | Arbre de règles enchaînées |

**Durée** : 1h30  
**Prérequis** : TP0 complété, `if/else`, fonctions Python

---

## Partie 1 — Logique booléenne

### 1.1 Rappel : les opérateurs booléens

En Python, on combine des conditions avec :
- `and` → vrai si **les deux** conditions sont vraies
- `or`  → vrai si **au moins une** condition est vraie
- `not` → **inverse** une condition

```python
x = 5
x > 3 and x < 10   # True  (5 est entre 3 et 10)
x > 3 and x < 4    # False (5 n'est pas < 4)
x > 3 or  x < 4    # True  (5 > 3, donc au moins une vraie)
not (x > 3)        # False (inverse de True)
```

### 1.2 Application : analyser une connexion réseau

Un paquet réseau est décrit par 4 attributs :
- `port` : port de destination (ex: 22 = SSH, 80 = HTTP, 443 = HTTPS)
- `paquets_par_sec` : nombre de paquets envoyés par seconde
- `taille_paquet` : taille moyenne des paquets en octets
- `ip_connue` : l'adresse IP source est-elle dans la liste blanche ? (True/False)

In [ ]:
# Une connexion réseau représentée par un dictionnaire
connexion = {
    'port'            : 22,     # SSH
    'paquets_par_sec' : 150,    # très élevé
    'taille_paquet'   : 40,     # petits paquets (typique scan de port)
    'ip_connue'       : False   # IP inconnue
}

print("Connexion à analyser :")
for cle, valeur in connexion.items():
    print(f"  {cle:20s} = {valeur}")

### 1.3 À vous — détecteur par logique booléenne

Complétez les conditions suivantes.  
Règles métier (définies par l'expert sécurité) :
- **Scan de port** : plus de 100 paquets/sec ET paquets < 60 octets
- **Brute force SSH** : port 22 ET IP inconnue ET plus de 50 paquets/sec
- **Connexion suspecte** : scan de port OU brute force SSH

In [ ]:
port             = connexion['port']
paquets_par_sec  = connexion['paquets_par_sec']
taille_paquet    = connexion['taille_paquet']
ip_connue        = connexion['ip_connue']

# Règle 1 : scan de port
scan_de_port = (paquets_par_sec > ???) and (taille_paquet < ???)   # ← complétez

# Règle 2 : brute force SSH
brute_force_ssh = (port == ???) and (not ???) and (paquets_par_sec > ???)   # ← complétez

# Règle 3 : connexion suspecte (combinaison)
suspecte = ??? or ???   # ← complétez

print(f"Scan de port     : {scan_de_port}")
print(f"Brute force SSH  : {brute_force_ssh}")
print(f"Connexion suspecte : {suspecte}")

### 1.4 Testez avec d'autres connexions

Modifiez les valeurs de `connexion` et relancez la cellule 1.3.  
Vérifiez que votre détecteur réagit correctement.

In [ ]:
# Testez ces 3 connexions une par une en modifiant `connexion` ci-dessus

# Connexion A — navigation web normale
# port=443, paquets_par_sec=5, taille_paquet=500, ip_connue=True
# → attendu : AUCUNE alerte

# Connexion B — scan de port agressif
# port=80, paquets_par_sec=200, taille_paquet=30, ip_connue=False
# → attendu : scan_de_port = True

# Connexion C — tentative SSH depuis IP inconnue (lente)
# port=22, paquets_par_sec=10, taille_paquet=200, ip_connue=False
# → attendu : brute_force_ssh = False (trop lent pour être du brute force)

print("Modifiez la cellule 'connexion' au-dessus et relancez la cellule 1.3")

---
## Partie 2 — Système expert : règles SI...ALORS

### 2.1 Qu'est-ce qu'un système expert ?

Un **système expert** encode le savoir d'un expert humain sous forme de règles :  

```
SI   <condition>   ALORS   <conclusion>   [SINON <autre conclusion>]
```

Exemple réel (Snort, un IDS open source) :
```
SI  paquet TCP  ET  port_dst=22  ET  contenu contient "Failed password"
ALORS  ALERTE "Tentative de brute force SSH"
```

**Avantage** : les règles sont lisibles et explicables par un humain.  
**Limite** : il faut écrire toutes les règles à la main — impossible si les cas sont trop nombreux.

### 2.2 Architecture d'un système expert simple

```
┌─────────────────┐      ┌──────────────────┐      ┌──────────────┐
│  Base de faits  │─────▶│  Moteur          │─────▶│  Conclusion  │
│  (la connexion) │      │  d'inférence     │      │  (alerte ?)  │
└─────────────────┘      │  (vos règles)    │      └──────────────┘
                         └──────────────────┘
```

In [ ]:
# Base de faits : les informations disponibles sur une connexion
faits = {
    'protocole'       : 'TCP',
    'port'            : 22,
    'paquets_par_sec' : 80,
    'taille_paquet'   : 45,
    'ip_connue'       : False,
    'heure'           : 3,       # 3h du matin
    'pays_source'     : 'CN',    # code pays
}

print("Base de faits chargée :")
for k, v in faits.items():
    print(f"  {k:20s} = {v}")

### 2.3 À vous — écrire les règles du système expert

Complétez chaque règle. Le système doit retourner une liste d'alertes déclenchées.

In [ ]:
def systeme_expert(faits):
    """
    Système expert de détection d'intrusion.
    Retourne la liste des alertes déclenchées.
    """
    alertes = []   # liste vide au départ
    
    # Extraction des faits pour plus de lisibilité
    protocole       = faits['protocole']
    port            = faits['port']
    paquets_par_sec = faits['paquets_par_sec']
    taille_paquet   = faits['taille_paquet']
    ip_connue       = faits['ip_connue']
    heure           = faits['heure']
    pays_source     = faits['pays_source']
    
    # ── Règle 1 : scan de port ───────────────────────────────────────────
    # SI paquets > 100 ET taille < 60
    # ALORS alerte "Scan de port détecté"
    if ??? and ???:   # ← complétez
        alertes.append("Scan de port détecté")
    
    # ── Règle 2 : brute force SSH ────────────────────────────────────────
    # SI protocole TCP ET port 22 ET IP inconnue ET paquets > 50
    # ALORS alerte "Brute force SSH"
    if ??? and ??? and ??? and ???:   # ← complétez
        alertes.append("Brute force SSH")
    
    # ── Règle 3 : activité nocturne suspecte ─────────────────────────────
    # SI heure entre 0h et 5h ET IP inconnue
    # ALORS alerte "Activité nocturne suspecte"
    if ??? and ???:   # ← complétez (heure >= 0 et heure <= 5)
        alertes.append("Activité nocturne suspecte")
    
    # ── Règle 4 : origine géographique à risque ──────────────────────────
    # SI pays source dans la liste à risque ET IP inconnue
    # ALORS alerte "Origine géographique à risque"
    pays_a_risque = ['CN', 'RU', 'KP']   # liste exemple
    if ??? and ???:   # ← complétez
        alertes.append("Origine géographique à risque")
    
    # ── Règle 5 : combinaison critique ───────────────────────────────────
    # SI plus de 2 alertes déjà déclenchées
    # ALORS alerte "CRITIQUE : intrusion probable"
    if len(alertes) > ???:   # ← complétez
        alertes.append("CRITIQUE : intrusion probable")
    
    return alertes


# Test
alertes = systeme_expert(faits)

print(f"Nombre d'alertes déclenchées : {len(alertes)}")
for alerte in alertes:
    print(f"  ⚠ {alerte}")

if not alertes:
    print("  ✓ Aucune alerte — trafic normal")

### 2.4 Analyser un lot de connexions

Un vrai IDS analyse des milliers de connexions. Testons notre système sur plusieurs cas.

In [ ]:
# Lot de connexions à analyser
log_connexions = [
    {'protocole':'TCP', 'port':443, 'paquets_par_sec':8,   'taille_paquet':800, 'ip_connue':True,  'heure':14, 'pays_source':'FR'},
    {'protocole':'TCP', 'port':22,  'paquets_par_sec':90,  'taille_paquet':40,  'ip_connue':False, 'heure':3,  'pays_source':'CN'},
    {'protocole':'UDP', 'port':53,  'paquets_par_sec':200, 'taille_paquet':20,  'ip_connue':False, 'heure':22, 'pays_source':'RU'},
    {'protocole':'TCP', 'port':80,  'paquets_par_sec':12,  'taille_paquet':600, 'ip_connue':True,  'heure':10, 'pays_source':'DE'},
    {'protocole':'TCP', 'port':22,  'paquets_par_sec':5,   'taille_paquet':200, 'ip_connue':False, 'heure':2,  'pays_source':'KP'},
]

print(f"{'#':3s} {'Port':6s} {'Paq/s':7s} {'IP ok':6s} {'Heure':6s} {'Pays':5s} {'Alertes'}")
print("-" * 70)

for i, c in enumerate(log_connexions):
    alertes = systeme_expert(c)
    alerte_str = " | ".join(alertes) if alertes else "OK"
    print(f"{i+1:3d} {c['port']:6d} {c['paquets_par_sec']:7d} {str(c['ip_connue']):6s} {c['heure']:6d} {c['pays_source']:5s} {alerte_str}")

### 2.5 Réflexion

Répondez dans la cellule ci-dessous.

**Question 1** — La connexion #5 (KP, port 22, nuit) est-elle détectée comme critique ? Pourquoi ?

*Votre réponse :* ...

**Question 2** — Citez une situation où ce système expert génèrerait un **faux positif** (alerte sur trafic légitime).

*Votre réponse :* ...

**Question 3** — Pourquoi un attaquant pourrait-il facilement **contourner** ce système ?

*Votre réponse :* ...

---
## Partie 3 — Arbre de décision

### 3.1 Principe

Un **arbre de décision** enchaîne des questions binaires (oui/non) jusqu'à une conclusion.  
C'est exactement ce qu'un analyste SOC fait mentalement :

```
                    Port = 22 ?
                   /           \
                 OUI            NON
                 /               \
          IP connue ?         Paquets > 100 ?
          /       \            /          \
        OUI       NON        OUI          NON
         |         |          |            |
       NORMAL   SUSPECT    SUSPECT       NORMAL
```

### 3.2 Implémentation : un nœud de l'arbre

On représente l'arbre avec des **dictionnaires imbriqués** :
- `question` : la condition à tester
- `oui` : branche si la condition est vraie
- `non` : branche si la condition est fausse
- `resultat` : une feuille (conclusion finale)

In [ ]:
# Construction de l'arbre de décision
# Chaque nœud est un dictionnaire avec 'question', 'oui', 'non'
# Chaque feuille est un dictionnaire avec 'resultat'

arbre_ids = {
    'question': 'port_ssh',         # Question racine : est-ce du SSH ?
    'oui': {
        'question': 'ip_connue',    # Oui SSH → l'IP est-elle connue ?
        'oui': {
            'question': 'paquets_eleves',  # IP connue → beaucoup de paquets ?
            'oui': {'resultat': 'SUSPECT - SSH flood depuis IP connue'},
            'non': {'resultat': 'NORMAL - SSH légitime'}
        },
        'non': {
            'question': 'heure_nuit',  # IP inconnue → c'est la nuit ?
            'oui': {'resultat': 'CRITIQUE - SSH inconnu de nuit'},
            'non': {'resultat': 'SUSPECT - SSH depuis IP inconnue'}
        }
    },
    'non': {
        'question': 'paquets_eleves',  # Pas SSH → beaucoup de paquets ?
        'oui': {
            'question': 'petits_paquets',  # Beaucoup de paquets → petits ?
            'oui': {'resultat': 'SUSPECT - Scan de port probable'},
            'non': {'resultat': 'SUSPECT - Flood non-SSH'}
        },
        'non': {'resultat': 'NORMAL - Trafic standard'}
    }
}

print("Arbre de décision construit !")
print(f"Question racine : '{arbre_ids['question']}'")

### 3.3 À vous — le moteur de parcours de l'arbre

Complétez la fonction qui **parcourt** l'arbre pour une connexion donnée.

In [ ]:
def evaluer_question(question, connexion):
    """
    Évalue une question (nœud de l'arbre) sur une connexion.
    Retourne True ou False.
    """
    if question == 'port_ssh':
        return connexion['port'] == 22
    elif question == 'ip_connue':
        return connexion['ip_connue'] == True
    elif question == 'paquets_eleves':
        return connexion['paquets_par_sec'] > 50
    elif question == 'petits_paquets':
        return connexion['taille_paquet'] < 60
    elif question == 'heure_nuit':
        return connexion['heure'] >= 0 and connexion['heure'] <= 6
    else:
        return False


def parcourir_arbre(noeud, connexion, chemin=[]):
    """
    Parcourt l'arbre de décision de façon récursive.
    Retourne le résultat final et le chemin parcouru.
    """
    # Cas de base : on est sur une feuille (le nœud a un 'resultat')
    if 'resultat' in noeud:
        return noeud['resultat'], chemin
    
    # Cas récursif : on est sur un nœud de décision
    question = noeud['question']
    reponse  = evaluer_question(question, connexion)
    
    # On enregistre le chemin pour expliquer la décision
    chemin = chemin + [f"{question} → {'OUI' if reponse else 'NON'}"]
    
    # On descend dans la bonne branche
    if reponse:
        return parcourir_arbre(???, connexion, chemin)   # ← branche OUI
    else:
        return parcourir_arbre(???, connexion, chemin)   # ← branche NON


# Test sur la connexion de base
connexion_test = {
    'port': 22, 'paquets_par_sec': 80, 'taille_paquet': 45,
    'ip_connue': False, 'heure': 3
}

resultat, chemin = parcourir_arbre(arbre_ids, connexion_test)

print("Chemin parcouru dans l'arbre :")
for etape in chemin:
    print(f"  → {etape}")
print(f"\nConclusion : {resultat}")

### 3.4 Analyser le même lot avec l'arbre

In [ ]:
# Réutilisation du lot de connexions de la Partie 2
print(f"{'#':3s} {'Port':6s} {'IP ok':7s} {'Heure':6s} {'Résultat'}")
print("-" * 65)

for i, c in enumerate(log_connexions):
    resultat, _ = parcourir_arbre(arbre_ids, c)
    print(f"{i+1:3d} {c['port']:6d} {str(c['ip_connue']):7s} {c['heure']:6d} {resultat}")

### 3.5 Bonus — Ajouter une branche à l'arbre

Modifiez `arbre_ids` pour ajouter une nouvelle question :  
Dans la branche "Pas SSH / pas de flood", distinguez le port 443 (HTTPS normal) du reste.

In [ ]:
# Ajoutez d'abord la question 'port_https' dans evaluer_question
# puis modifiez la feuille 'NORMAL - Trafic standard' dans arbre_ids

# Votre code ici


---
## Partie 4 — Comparaison des approches

Complétez le tableau en vous basant sur ce que vous avez observé dans les 3 parties.

In [ ]:
print("+" + "-"*22 + "+" + "-"*22 + "+" + "-"*22 + "+")
print(f"| {'Critère':20s} | {'Logique booléenne':20s} | {'Système expert':20s} |")
print("+" + "="*22 + "+" + "="*22 + "+" + "="*22 + "+")
criteres = [
    ("Lisibilité",       "Simple",         "Bonne"),
    ("Explicabilité",    "Oui",             "Oui"),
    ("Passage à l'échelle", "Difficile",   "Difficile"),
    ("Adaptation auto",  "Non",             "Non"),
    ("Faux positifs",    "Selon seuils",    "Selon règles"),
]
for c, b, e in criteres:
    print(f"| {c:20s} | {b:20s} | {e:20s} |")
print("+" + "-"*22 + "+" + "-"*22 + "+" + "-"*22 + "+")
print("\nProchaine étape → TP0/TP1 : le perceptron apprend ces règles TOUT SEUL !")

---
## Récapitulatif

| Partie | Concept | Ce que vous avez fait |
|--------|---------|----------------------|
| 1 | Logique booléenne | `and`, `or`, `not` sur des attributs réseau |
| 2 | Système expert | Règles SI...ALORS, moteur d'inférence, lots |
| 3 | Arbre de décision | Parcours récursif, chemin explicable |

**La limite commune** : toutes ces règles sont écrites **à la main** par un expert.  
→ Le Machine Learning (TP1 et suivants) va les **apprendre automatiquement** depuis les données.

---
**Rendu** : sauvegardez (`Ctrl+S`) et déposez `tp_s1_NOM_Prenom.ipynb` sur le portail.